## Second experiment - second configuration.

In [ ]:
import sys
import csv
import os
import warnings
import glob
import matplotlib.backends.backend_pdf
import mne
import pandas as pd
from mne import Epochs, find_events, pick_events, pick_types
from collections import OrderedDict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from datetime import datetime
import numpy as np
from scipy import signal
from scipy.signal import decimate
from scipy.stats.mstats import winsorize
from mne.preprocessing import create_eog_epochs, create_ecg_epochs

from analysis_tools import load_raw

from sklearn.pipeline import make_pipeline

from sklearn.model_selection import train_test_split

from mne.decoding import Vectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA

from sklearn.model_selection import cross_val_score, StratifiedShuffleSplit, StratifiedKFold

from pyriemann.estimation import ERPCovariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from pyriemann.spatialfilters import Xdawn
import pandas as pd
import seaborn as sns
from datetime import datetime as dt

from collections import OrderedDict
from sklearn import datasets

from sklearn.metrics import roc_auc_score

from sklearn.metrics import roc_auc_score, accuracy_score

from sklearn.model_selection import cross_val_predict

from sklearn.svm import LinearSVC
from sklearn.svm import SVC

from sklearn.metrics import confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import SGDClassifier

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.svm import SVC

from sklearn.neighbors import KNeighborsClassifier

import random

from sklearn.multiclass import OneVsRestClassifier

### Application of Notch to attenuate the frequency at 50 Hz, the sixth-order Butterworth band-pass filter with cut-off frequencies of 1-17 Hz, and ICA. After their application, the framework generates the epochs.

In [ ]:
def process_by_subject(subject_name, plot_images):
    count = 1
    datasets = sorted(glob.glob('data/'+ subject_name + '_*.csv'))
    df_final = pd.DataFrame()
    array_epochs = []
    for dataset in datasets:
        sampling_rate = 256
        convert_raw_openbci_to_csv = False
        subject = 0
        session = 0
        ch_names = {}
        
        if convert_raw_openbci_to_csv:
            dataset = convert_new_openbci(dataset)
        
        raw = load_raw(dataset, sfreq=sampling_rate, stim_ind=8, replace_ch_names=None, ch_ind=[0, 1, 2, 3, 4, 5, 6, 7])
        
        for i, chn in enumerate(raw.ch_names):
            ch_names[chn] = i
                        
        sfreq = raw.info['sfreq']
        data_ini, times_ini = raw[:-1, int(sfreq * 1):int(sfreq * 10)]
        
        raw_notch = raw.copy().notch_filter([50.0])

        iir_params = dict(order=6, ftype='butter')
        raw_notch_and_filter = raw_notch.copy().filter(1, 17, method='iir', iir_params=iir_params)
            
        ica = mne.preprocessing.ICA(n_components=8, random_state=97)
        ica.fit(raw_notch_and_filter)
        
        raw_notch_and_filter_ica = raw_notch_and_filter.copy()
        
        ica.exclude = []
        eog_inds, eog_scores = ica.find_bads_eog(raw_notch_and_filter_ica, ['Fp1','Fp2'], threshold=1.5)
        ica.exclude = eog_inds
        
        ica.apply(raw_notch_and_filter_ica)

        events = find_events(raw_notch_and_filter_ica, shortest_event=1) 
        
        baseline = (None, 0)
        
        event_id = {'Target': 1, 'NoTarget': 2}
        reject = {'eeg': 100e-6}

        epochs = Epochs(raw_notch_and_filter_ica, events=events, event_id=event_id, tmin=-0.1, tmax=0.8, reject=reject, preload=True)
        epochs.pick_types(eeg=True)
    
        array_epochs.append(epochs)
        
        count = count + 1
    
    return mne.concatenate_epochs(array_epochs, add_offset=True)

process_by_subject("Carlos", False).save('epochs/epochs_Carlos_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Javi", False).save('epochs/epochs_Javi_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Kike", False).save('epochs/epochs_Kike_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Leo", False).save('epochs/epochs_Leo_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Lola", False).save('epochs/epochs_Lola_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Mariaje", False).save('epochs/epochs_Mariaje_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Mario", False).save('epochs/epochs_Mario_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Marivi", False).save('epochs/epochs_Marivi_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Mati", False).save('epochs/epochs_Mati_second_experiment_second_configuration-epo.fif', overwrite = True)
process_by_subject("Pedro", False).save('epochs/epochs_Pedro_second_experiment_second_configuration-epo.fif', overwrite = True)

### Generation of five authentication datasets.

In [ ]:
def get_epochs_authentication():    
    epochs_Carlos = mne.read_epochs('epochs/epochs_Carlos_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Javi = mne.read_epochs('epochs/epochs_Javi_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Kike = mne.read_epochs('epochs/epochs_Kike_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Leo = mne.read_epochs('epochs/epochs_Leo_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Lola = mne.read_epochs('epochs/epochs_Lola_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Mariaje = mne.read_epochs('epochs/epochs_Mariaje_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Mario = mne.read_epochs('epochs/epochs_Mario_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Marivi = mne.read_epochs('epochs/epochs_Marivi_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Mati = mne.read_epochs('epochs/epochs_Mati_second_experiment_second_configuration-epo.fif', preload=False)
    epochs_Pedro = mne.read_epochs('epochs/epochs_Pedro_second_experiment_second_configuration-epo.fif', preload=False)
    
    subjects = ["Carlos", "Javi", "Kike", "Leo", "Lola", "Mariaje", "Mario", "Marivi", "Mati", "Pedro"]
    epochs_subjects = [epochs_Carlos, epochs_Javi, epochs_Kike, epochs_Leo, epochs_Lola, epochs_Mariaje, epochs_Mario, epochs_Marivi, epochs_Mati, epochs_Pedro]
   
    y_names = []
    subject = 0
    for epochs in epochs_subjects:
        no_targets = np.count_nonzero(epochs.events[:, -1]==2)
        
        index_no_targets = []
        y = epochs.events[:, -1]

        while(no_targets != 0):
            position = random.randint(0, len(y)-1)
            if y[position] == 2 and position not in index_no_targets:
                index_no_targets.append(position)
                no_targets -= 1

        epochs.drop(index_no_targets)
    
        subject += 1
        
    minimun_targets = 10000
    
    for epochs in epochs_subjects:
        targets = np.count_nonzero(epochs.events[:, -1]==1)
        if (targets < minimun_targets):
            minimun_targets = targets
    
    for epochs in epochs_subjects:
        index_leftover_targets = []
        targets = np.count_nonzero(epochs.events[:, -1]==1)
        y = epochs.events[:, -1]
        targets_out = targets - minimun_targets
        while(targets_out != 0):
            position = random.randint(0, len(y)-1)
            if position not in index_leftover_targets:
                index_leftover_targets.append(position)
                targets_out -= 1
                
        epochs.drop(index_leftover_targets)
                           
    subject = 0
    for epochs in epochs_subjects:
        targets = np.count_nonzero(epochs.events[:, -1]==1)
        name_subject = subjects[subject]
        
        for i in range (targets):
            y_names = np.append(y_names, name_subject)
                  
        subject += 1
                
    return mne.concatenate_epochs(epochs_subjects, add_offset=True), y_names;
    
for i in range(5):
        epochs_caso_base_authentication_multiclass, y = get_epochs_authentication()
        epochs_caso_base_authentication_multiclass.save('epochs/epochs_{}_authentication_second_experiment_second_configuration-epo.fif'.format(i), overwrite = True)

df = pd.DataFrame(y)
df.to_csv('epochs/y_authentication_second_experiment_second_configuration.csv', index=False)

### Generation of a CSV file that will contain the results obtained in the authentication process.

In [ ]:
header = ['Option/Classifier', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
with open('second_experiment_second_configuration.csv', 'w', encoding='UTF8') as f:
    writer = csv.writer(f)
    
    writer.writerow(header)

### Authentication process using multiclass classification.

In [ ]:
clfs = OrderedDict()

clfs['Clasificador I'] = make_pipeline(Vectorizer(), StandardScaler(), LogisticRegression())
clfs['Clasificador II'] = make_pipeline(Vectorizer(), LDA(shrinkage='auto', solver='eigen'))
clfs['Clasificador III'] = make_pipeline(Xdawn(2, classes=[1]), Vectorizer(), LDA(shrinkage='auto', solver='eigen'))
clfs['Clasificador IV'] = make_pipeline(ERPCovariances(estimator='oas'), TangentSpace(), LogisticRegression())
clfs['Clasificador V'] = make_pipeline(ERPCovariances(estimator='oas'), MDM())
clfs['Clasificador VI'] = make_pipeline(Vectorizer(), RandomForestClassifier(random_state=42))
clfs['Clasificador VII'] = make_pipeline(Vectorizer(), QDA())
clfs['Clasificador VIII'] = make_pipeline(Vectorizer(), SVC(gamma='scale'))
clfs['Clasificador IX'] = make_pipeline(Vectorizer(), KNeighborsClassifier(n_neighbors=50))
clfs['Clasificador X'] = make_pipeline(Xdawn(2, classes=[1]), Vectorizer(), RandomForestClassifier(random_state=42))
clfs['Clasificador XI'] = make_pipeline(ERPCovariances(estimator='oas'), TangentSpace(), RandomForestClassifier(random_state=42))
clfs['Clasificador XII'] = make_pipeline(ERPCovariances(estimator='oas'), Vectorizer(), RandomForestClassifier(random_state=42))
clfs['Clasificador XIII'] = make_pipeline(Xdawn(2, classes=[1]), Vectorizer(), QDA())
clfs['Clasificador XIV'] = make_pipeline(ERPCovariances(estimator='oas'), TangentSpace(), QDA())
clfs['Clasificador XV'] = make_pipeline(ERPCovariances(estimator='oas'), Vectorizer(), QDA())
clfs['Clasificador XVI'] = make_pipeline(Xdawn(2, classes=[1]), Vectorizer(), SVC(gamma='scale'))
clfs['Clasificador XVII'] = make_pipeline(ERPCovariances(estimator='oas'), TangentSpace(), SVC(gamma='scale'))
clfs['Clasificador XVIII'] = make_pipeline(ERPCovariances(estimator='oas'), Vectorizer(), SVC(gamma='scale'))
clfs['Clasificador XIX'] = make_pipeline(Xdawn(2, classes=[1]), Vectorizer(), KNeighborsClassifier(n_neighbors=50))
clfs['Clasificador XX'] = make_pipeline(ERPCovariances(estimator='oas'), TangentSpace(), KNeighborsClassifier(n_neighbors=50))
clfs['Clasificador XXI'] = make_pipeline(ERPCovariances(estimator='oas'), Vectorizer(), KNeighborsClassifier(n_neighbors=50))


def authentication_multiclass(epochs, y_all_subjects, experiment):
   
    option = 'Second_experiment_second_configuration_' + str(experiment)
    
    data = []
    data.append(option)
    
    X = epochs.get_data() * 1e6
    times = epochs.times
    y = y_all_subjects
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    auc = []
    methods = []
    for m in clfs:
        clf = OneVsRestClassifier(clfs[m])
        clf.fit(X_train, y_train)
        score = clf.predict(X_test)
        report = classification_report(y_test, score, output_dict=True)
        accuracy = report['accuracy']
        accuracy = round(accuracy, 2)
        data.append(accuracy)
        

    with open('second_experiment_second_configuration.csv', 'a') as f:
        writer = csv.writer(f)
    
        writer.writerow(data)
        
        f.close()
      
    print("Fin del proceso numero {} de autenticación multiclase:". format(experiment), dt.now().strftime('%d-%m-%Y, %H:%M:%S (GMT+1)'))
        
df = pd.read_csv('epochs/y_authentication_second_experiment_second_configuration.csv')
y_names = df['0']

for i in range(5):
    authentication_multiclass(mne.read_epochs('epochs/epochs_{}_authentication_second_experiment_second_configuration-epo.fif'.format(i), preload=False), y_names, i)